### Step 1 — Authenticate & Configure Parameters (~10s) | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/luiscuellarh/apigee-unified-ai-gateway/blob/main/00_setup_vpc_and_ip_ranges.ipynb)
**Critical Concept:** The reserved `/22` and `/28` are **GLOBAL allocations attached to the NETWORK**, not subdivisions of the subnet. They must not overlap `SUBNET_CIDR`, nothing of yours is ever deployed inside them (they are consumed by the Google-managed Apigee tenant project), and you must **never create a subnetwork inside a reserved range**.
```text
[VPC: apigee-network] ──► [DEPLOY (regional subnet): 10.10.0.0/24]  |  [RESERVE (global runtime): 10.20.0.0/22]  |  [RESERVE (global support): 10.30.0.0/28]
```

In [ ]:
#@title 1. Authenticate & Configure Parameters { display-mode: "form" }
from google.colab import auth

PROJECT_ID         = "" #@param {type:"string"}
REGION             = "europe-west1" #@param {type:"string"}
NETWORK_NAME       = "apigee-network" #@param {type:"string"}
SUBNET_NAME        = "apigee-subnet" #@param {type:"string"}
SUBNET_CIDR        = "10.10.0.0/24" #@param {type:"string"}
PEERING_RANGE_NAME = "apigee-peering-range" #@param {type:"string"}
PEERING_CIDR_SIZE  = "/22" #@param ["/22", "/28"]
PEERING_RANGE_CIDR = "10.20.0.0/22" #@param {type:"string"}
SUPPORT_RANGE_NAME = "apigee-support-range" #@param {type:"string"}
SUPPORT_RANGE_CIDR = "10.30.0.0/28" #@param {type:"string"}

auth.authenticate_user()
!gcloud config set project $PROJECT_ID --quiet
print(f"✅ Authenticated and configured gcloud project '{PROJECT_ID}'")

### Step 2 — Pre-Flight Validation & Resource Existence Checks (~10–20s)
Verifies that all required variables are set, checks access to `PROJECT_ID`, and enables `compute.googleapis.com` and `servicenetworking.googleapis.com` if not already enabled.
Validates RFC1918 compliance, prefix lengths, and 3-way CIDR non-overlap using `ipaddress`, then checks whether each of the 4 resources already exists with matching configuration (`ℹ️ reuse`) or conflicting configuration (`❌ stop`) before printing the `✅` execution plan table.

In [ ]:
#@title 2. Pre-Flight Validation & Resource Existence Checks
import ipaddress
import json
import subprocess

# a) Every required variable is defined and non-empty (report all missing at once)
REQUIRED_VARS = [
    "PROJECT_ID", "REGION", "NETWORK_NAME", "SUBNET_NAME", "SUBNET_CIDR",
    "PEERING_RANGE_NAME", "PEERING_CIDR_SIZE", "PEERING_RANGE_CIDR",
    "SUPPORT_RANGE_NAME", "SUPPORT_RANGE_CIDR",
]
missing = [v for v in REQUIRED_VARS if not str(globals().get(v, "")).strip()]
if missing:
    raise SystemExit(f"❌ Missing or empty required variables: {', '.join(missing)}")

# b) PROJECT_ID resolves and caller has access
proj_res = subprocess.run(
    ["gcloud", "projects", "describe", PROJECT_ID, "--format=json"],
    capture_output=True, text=True,
)
if proj_res.returncode != 0:
    raise SystemExit(f"❌ Cannot access PROJECT_ID '{PROJECT_ID}': {proj_res.stderr.strip()}")
print(f"✅ Verified caller access to project '{PROJECT_ID}'")

# c) Ensure compute.googleapis.com and servicenetworking.googleapis.com are enabled
REQUIRED_APIS = {"compute.googleapis.com", "servicenetworking.googleapis.com"}
enabled_apis = set(subprocess.check_output(
    ["gcloud", "services", "list", "--enabled", f"--project={PROJECT_ID}", "--format=value(config.name)"],
    text=True,
).splitlines())
missing_apis = sorted(REQUIRED_APIS - enabled_apis)
if missing_apis:
    !gcloud services enable {" ".join(missing_apis)} --project=$PROJECT_ID --quiet
    print(f"✅ Enabled APIs: {', '.join(missing_apis)}")
else:
    print("ℹ️ Required APIs already enabled (compute.googleapis.com, servicenetworking.googleapis.com)")

# d) Parse every CIDR with ipaddress and confirm RFC1918 private space + prefix lengths
RFC1918 = (
    ipaddress.IPv4Network("10.0.0.0/8"),
    ipaddress.IPv4Network("172.16.0.0/12"),
    ipaddress.IPv4Network("192.168.0.0/16"),
)
parsed_blocks = {}
for label, raw_cidr in [
    ("SUBNET_CIDR", SUBNET_CIDR),
    ("PEERING_RANGE_CIDR", PEERING_RANGE_CIDR),
    ("SUPPORT_RANGE_CIDR", SUPPORT_RANGE_CIDR),
]:
    if "/" not in raw_cidr:
        raise SystemExit(f"❌ {label} ('{raw_cidr}') must include a '/prefix' length.")
    net = ipaddress.ip_network(raw_cidr.strip(), strict=True)
    if not isinstance(net, ipaddress.IPv4Network) or not any(net.subnet_of(b) for b in RFC1918):
        raise SystemExit(f"❌ {label} ('{net}') must be an RFC1918 private IPv4 network (10/8, 172.16/12, 192.168/16).")
    parsed_blocks[label] = net

subnet_net  = parsed_blocks["SUBNET_CIDR"]
peering_net = parsed_blocks["PEERING_RANGE_CIDR"]
support_net = parsed_blocks["SUPPORT_RANGE_CIDR"]

expected_peering_prefix = int(PEERING_CIDR_SIZE.strip().lstrip("/"))
if peering_net.prefixlen != expected_peering_prefix:
    raise SystemExit(
        f"❌ Prefix length mismatch: PEERING_RANGE_CIDR ('{peering_net}') is /{peering_net.prefixlen}, "
        f"but PEERING_CIDR_SIZE is '{PEERING_CIDR_SIZE}'."
    )
if support_net.prefixlen != 28:
    raise SystemExit(f"❌ Support range must be exactly /28, but SUPPORT_RANGE_CIDR ('{support_net}') is /{support_net.prefixlen}.")

# e) Confirm the 3 blocks are mutually non-overlapping
for name_a, name_b, net_a, net_b in [
    (f"SUBNET_CIDR ({subnet_net})", f"PEERING_RANGE_CIDR ({peering_net})", subnet_net, peering_net),
    (f"SUBNET_CIDR ({subnet_net})", f"SUPPORT_RANGE_CIDR ({support_net})", subnet_net, support_net),
    (f"PEERING_RANGE_CIDR ({peering_net})", f"SUPPORT_RANGE_CIDR ({support_net})", peering_net, support_net),
]:
    if net_a.overlaps(net_b):
        raise SystemExit(f"❌ CIDR collision: {name_a} overlaps with {name_b}.")

# f) Pre-existence & config-match checks for the 4 resources + project-wide overlap scan
nets_json = json.loads(subprocess.check_output(
    ["gcloud", "compute", "networks", "list", f"--project={PROJECT_ID}", "--format=json"], text=True
) or "[]")
subs_json = json.loads(subprocess.check_output(
    ["gcloud", "compute", "networks", "subnets", "list", f"--project={PROJECT_ID}", "--format=json"], text=True
) or "[]")
addrs_json = json.loads(subprocess.check_output(
    ["gcloud", "compute", "addresses", "list", "--global", f"--project={PROJECT_ID}", "--format=json"], text=True
) or "[]")

# 1. VPC Network existence check
existing_net = next((n for n in nets_json if n.get("name") == NETWORK_NAME), None)
if existing_net is None:
    CREATE_NETWORK, net_action = True, "✅ create it"
elif existing_net.get("autoCreateSubnetworks") is False:
    CREATE_NETWORK, net_action = False, "ℹ️ reuse, skip creation"
else:
    raise SystemExit(
        f"❌ Network '{NETWORK_NAME}' exists with DIFFERENT config: "
        f"expected autoCreateSubnetworks=False (custom mode), got autoCreateSubnetworks={existing_net.get('autoCreateSubnetworks')}."
    )

# 2. Regional Subnet existence check & project subnet overlap scan
existing_sub = next(
    (s for s in subs_json if s.get("name") == SUBNET_NAME and s.get("region", "").endswith(f"/{REGION}")),
    None,
)
if existing_sub is None:
    CREATE_SUBNET, sub_action = True, "✅ create it"
else:
    actual_sub_net = existing_sub.get("network", "").split("/")[-1]
    actual_sub_cidr = existing_sub.get("ipCidrRange", "")
    if actual_sub_net != NETWORK_NAME or actual_sub_cidr != str(subnet_net):
        raise SystemExit(
            f"❌ Subnet '{SUBNET_NAME}' in '{REGION}' exists with DIFFERENT config: "
            f"expected (network={NETWORK_NAME}, cidr={subnet_net}), got (network={actual_sub_net}, cidr={actual_sub_cidr})."
        )
    CREATE_SUBNET, sub_action = False, "ℹ️ reuse, skip creation"

for s in subs_json:
    s_name, s_reg, s_cidr = s.get("name", ""), s.get("region", "").split("/")[-1], s.get("ipCidrRange")
    if not s_cidr:
        continue
    s_net = ipaddress.IPv4Network(s_cidr, strict=False)
    for blk_name, blk_net in [
        (f"SUBNET_CIDR ({subnet_net})", subnet_net),
        (f"PEERING_RANGE_CIDR ({peering_net})", peering_net),
        (f"SUPPORT_RANGE_CIDR ({support_net})", support_net),
    ]:
        if blk_net == subnet_net and s_name == SUBNET_NAME and s_reg == REGION and not CREATE_SUBNET:
            continue
        if blk_net.overlaps(s_net):
            raise SystemExit(f"❌ CIDR collision: {blk_name} overlaps existing subnet '{s_name}' ({s_net}) in '{s_reg}'.")

# 3 & 4. Global Reserved Ranges existence check & project global address overlap scan
range_decisions = {}
for r_name, exp_net in [(PEERING_RANGE_NAME, peering_net), (SUPPORT_RANGE_NAME, support_net)]:
    match = next((a for a in addrs_json if a.get("name") == r_name), None)
    if match is None:
        range_decisions[r_name] = (True, "✅ create it")
    else:
        act_purpose = match.get("purpose", "")
        act_net = match.get("network", "").split("/")[-1]
        act_cidr = f"{match.get('address', '')}/{match.get('prefixLength', '')}"
        if act_purpose != "VPC_PEERING" or act_net != NETWORK_NAME or act_cidr != str(exp_net):
            raise SystemExit(
                f"❌ Global range '{r_name}' exists with DIFFERENT config: "
                f"expected (purpose=VPC_PEERING, network={NETWORK_NAME}, cidr={exp_net}), "
                f"got (purpose={act_purpose}, network={act_net}, cidr={act_cidr})."
            )
        range_decisions[r_name] = (False, "ℹ️ reuse, skip creation")

CREATE_PEERING_RANGE, peering_action = range_decisions[PEERING_RANGE_NAME]
CREATE_SUPPORT_RANGE, support_action = range_decisions[SUPPORT_RANGE_NAME]

for a in addrs_json:
    a_name, a_ip, a_plen = a.get("name", ""), a.get("address"), a.get("prefixLength")
    if not a_ip:
        continue
    a_net = ipaddress.IPv4Network(f"{a_ip}/{a_plen}" if a_plen is not None else f"{a_ip}/32", strict=False)
    for blk_label, blk_res_name, blk_net in [
        ("SUBNET_CIDR", SUBNET_NAME, subnet_net),
        ("PEERING_RANGE_CIDR", PEERING_RANGE_NAME, peering_net),
        ("SUPPORT_RANGE_CIDR", SUPPORT_RANGE_NAME, support_net),
    ]:
        if a_name == blk_res_name and a_net == blk_net:
            continue
        if blk_net.overlaps(a_net):
            raise SystemExit(f"❌ CIDR collision: {blk_label} ({blk_net}) overlaps existing global address '{a_name}' ({a_net}).")

# g) Print the ✅ validated plan & existence table
rows = [
    (NETWORK_NAME,       "VPC (custom mode)", "global",             "subnet-mode=custom", net_action),
    (SUBNET_NAME,        "DEPLOY (regional)", REGION,               str(subnet_net),      sub_action),
    (PEERING_RANGE_NAME, "RESERVE (global)",  "global (no subnet)", str(peering_net),     peering_action),
    (SUPPORT_RANGE_NAME, "RESERVE (global)",  "global (no subnet)", str(support_net),     support_action),
]
print("\n✅ PRE-FLIGHT VALIDATION PASSED — EXECUTION PLAN:")
hdr = f"{'Resource':<22} | {'Type':<18} | {'Scope':<18} | {'CIDR / Mode':<20} | {'Existence / Action'}"
print("-" * len(hdr))
print(hdr)
print("-" * len(hdr))
for r_name, r_type, r_scope, r_cidr, r_act in rows:
    print(f"{r_name:<22} | {r_type:<18} | {r_scope:<18} | {r_cidr:<20} | {r_act}")
print("-" * len(hdr))

### Step 3 — Create VPC, Deploy Regional Subnet & Reserve Global Ranges (~30–45s)
Creates only the resources marked `✅ create it` by Step 2 and skips any that already exist with matching configuration (`ℹ️`).
Runs without `|| true` so any real `gcloud` failure surfaces immediately, then calls `describe` on both global ranges to print their actual allocated CIDRs.

In [ ]:
#@title 3. Create VPC, Deploy Regional Subnet & Reserve Both Global Ranges

if CREATE_NETWORK:
    !gcloud compute networks create $NETWORK_NAME --project=$PROJECT_ID --subnet-mode=custom --quiet
    print(f"✅ Created custom-mode VPC network '{NETWORK_NAME}'")
else:
    print(f"ℹ️ Reusing existing custom-mode VPC network '{NETWORK_NAME}'")

if CREATE_SUBNET:
    !gcloud compute networks subnets create $SUBNET_NAME --project=$PROJECT_ID --network=$NETWORK_NAME --region=$REGION --range=$SUBNET_CIDR --enable-private-ip-google-access --enable-flow-logs --quiet
    print(f"✅ Deployed regional subnet '{SUBNET_NAME}' ({SUBNET_CIDR}) in {REGION}")
else:
    print(f"ℹ️ Reusing existing regional subnet '{SUBNET_NAME}' ({SUBNET_CIDR}) in {REGION}")

peering_base_ip = str(peering_net.network_address)
peering_prefix  = peering_net.prefixlen
if CREATE_PEERING_RANGE:
    !gcloud compute addresses create $PEERING_RANGE_NAME --global --project=$PROJECT_ID --purpose=VPC_PEERING --network=$NETWORK_NAME --addresses=$peering_base_ip --prefix-length=$peering_prefix --quiet
    print(f"✅ Reserved global runtime range '{PEERING_RANGE_NAME}' ({peering_net})")
else:
    print(f"ℹ️ Reusing existing global runtime range '{PEERING_RANGE_NAME}' ({peering_net})")

support_base_ip = str(support_net.network_address)
if CREATE_SUPPORT_RANGE:
    !gcloud compute addresses create $SUPPORT_RANGE_NAME --global --project=$PROJECT_ID --purpose=VPC_PEERING --network=$NETWORK_NAME --addresses=$support_base_ip --prefix-length=28 --quiet
    print(f"✅ Reserved global support range '{SUPPORT_RANGE_NAME}' ({support_net})")
else:
    print(f"ℹ️ Reusing existing global support range '{SUPPORT_RANGE_NAME}' ({support_net})")

# Describe both reserved ranges and print their actual allocated CIDRs
p_desc = json.loads(subprocess.check_output(
    ["gcloud", "compute", "addresses", "describe", PEERING_RANGE_NAME, "--global", f"--project={PROJECT_ID}", "--format=json"], text=True
))
s_desc = json.loads(subprocess.check_output(
    ["gcloud", "compute", "addresses", "describe", SUPPORT_RANGE_NAME, "--global", f"--project={PROJECT_ID}", "--format=json"], text=True
))
ACTUAL_PEERING_CIDR = f"{p_desc['address']}/{p_desc['prefixLength']}"
ACTUAL_SUPPORT_CIDR = f"{s_desc['address']}/{s_desc['prefixLength']}"
print(f"✅ Described reserved global ranges — actual allocated CIDRs: {PEERING_RANGE_NAME}={ACTUAL_PEERING_CIDR}, {SUPPORT_RANGE_NAME}={ACTUAL_SUPPORT_CIDR}")

### Step 4 — Handoff Summary (< 5s)
Prints the **`DEPLOYED (regional)`** and **`RESERVED (global)`** summary sections and the exact variable values to paste into the Apigee provisioning notebook.
> ℹ️ **Service Networking Peering Note:** The Service Networking peering is established during **Apigee org creation**, not here. If org creation fails later, verify peering with:
> `gcloud services vpc-peerings list --network=$NETWORK_NAME`

In [ ]:
#@title 4. Handoff Summary
print(f"""======================================================================
DEPLOYED (regional):
  Network : {NETWORK_NAME} (custom subnet mode)
  Subnet  : {SUBNET_NAME}
  Region  : {REGION}
  CIDR    : {SUBNET_CIDR}

RESERVED (global):
  Runtime Range : {PEERING_RANGE_NAME} ({ACTUAL_PEERING_CIDR})
  Support Range : {SUPPORT_RANGE_NAME} ({ACTUAL_SUPPORT_CIDR})
======================================================================

# --- Paste into the Apigee X Provisioning Notebook ---
PROJECT_ID         = "{PROJECT_ID}"
REGION             = "{REGION}"
NETWORK_NAME       = "{NETWORK_NAME}"
SUBNET_NAME        = "{SUBNET_NAME}"
SUBNET_CIDR        = "{SUBNET_CIDR}"
PEERING_RANGE_NAME = "{PEERING_RANGE_NAME}"
PEERING_RANGE_CIDR = "{ACTUAL_PEERING_CIDR}"
SUPPORT_RANGE_NAME = "{SUPPORT_RANGE_NAME}"
SUPPORT_RANGE_CIDR = "{ACTUAL_SUPPORT_CIDR}"

# Note: Service Networking peering is established during Apigee org creation, not here.
""")